In [3]:
import os
import yaml

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

os.environ['GOOGLE_API_KEY'] = config['google']['api']
os.environ['SERP_API_KEY'] = config['serp']['api']

In [2]:
import logging
import traceback
import uuid
import gradio as gr
import re
import os
import sys
import spacy
import torch
import pandas as pd
import pickle
import requests
import json
import threading
import nest_asyncio
import uvicorn
import time
import socket 
from rapidfuzz import fuzz
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertModel
from google.adk.tools.agent_tool import AgentTool
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.a2a.utils.agent_to_a2a import to_a2a

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

sys.path.append('../utils/')
from mxnet_utils import BERTClassifier, CustomVocab

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nlp = spacy.load("en_core_web_md")
analyzer = SentimentIntensityAnalyzer()

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

label_map = vocabs["label_map"]
topic_vocab = vocabs["topic_vocab"]
author_vocab = vocabs["author_vocab"]
job_vocab = vocabs["job_vocab"]
location_vocab = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

spam_tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")
spam_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_base = BertModel.from_pretrained("bert-base-uncased")

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192,
)
net_best2.load_state_dict(torch.load("../checkpoints/best_cat.pth", map_location=device))
net_best2.to(device)
net_best2.eval()

def func_political_bias(text: str) -> str:
    statistic_types = {"CARDINAL", "PERCENT", "MONEY", "QUANTITY"}
    doc = nlp(str(text))
    stat_count = sum(ent.label_ in statistic_types for ent in doc.ents)
    def count_matches(stmt, bigram_list):
        words = [w.text.lower() for w in nlp(str(stmt))]
        if len(words) < 2: return 0
        bigrams = ["".join(words[i:i+2]) for i in range(len(words)-1)]
        matches = 0
        for bg in bigrams:
            for check in bigram_list:
                if fuzz.ratio(bg, check) >= 70:
                    matches += 1
                    break
        return matches
    return json.dumps({
        "stat_density": stat_count,
        "conservative_talking_points": count_matches(text, conservative_bigrams),
        "liberal_talking_points": count_matches(text, liberal_bigrams),
    })

def func_sensationalism(text: str) -> str:
    score = analyzer.polarity_scores(str(text))["compound"]
    return json.dumps({"emotional_intensity": abs(score), "polarity": score})

def func_spam(text: str) -> str:
    inputs = spam_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    return json.dumps({"spam_probability": probs[0,1].item()})

def func_BERT(text: str) -> str:
    split_doc = nlp(str(text))
    sentences = [sent.text.strip() for sent in split_doc.sents if sent.text.strip()]
    rev = {v: k for k, v in label_map.items()}
    prob_list = []
    for sent in sentences:
        enc = bert_tokenizer(str(sent), return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
        with torch.no_grad():
            outputs = net_best2(enc["input_ids"], enc.get("token_type_ids", torch.zeros_like(enc["input_ids"])).to(device), enc["attention_mask"],
                torch.zeros(1, len(topic_vocab)).to(device), torch.tensor([0]).to(device), torch.tensor([0]).to(device),
                torch.tensor([0]).to(device), torch.tensor([0]).to(device), torch.zeros(1, len(label_map)).to(device))
            probs = torch.softmax(outputs, dim=1)
            prob_list.append(probs[0].cpu()) 
    avg_probs = torch.stack(prob_list).mean(dim=0)
    pred = torch.argmax(avg_probs).item()
    return json.dumps({"model_prediction": rev[pred], "confidence": avg_probs[pred].item(), "class_probabilities": {rev[i]: avg_probs[i].item() for i in range(len(label_map))}})

def func_web_search(text: str) -> str:
    url = "https://serpapi.com/search"
    params = {"q": text, "api_key": os.environ.get("SERP_API_KEY"), "engine": "google", "num": 4}
    try:
        res = requests.get(url, params=params).json()
        evidence = [res["answer_box"].get("answer", res["answer_box"].get("snippet"))] if "answer_box" in res else []
        for item in res.get("organic_results", []): evidence.append(item.get("snippet"))
        return "\n".join(evidence) if evidence else "No live evidence found."
    except: return "Search Error"

worker_llm = LiteLlm(model="gemini/gemini-3-flash-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
manager_llm = LiteLlm(model="gemini/gemini-3-pro-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
worker_instruction = "Call your tool immediately with the text you receive. Return only the tool output."

bias_agent = LlmAgent(name="Political_Bias_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_political_bias])
sensational_agent = LlmAgent(name="Sensationalism_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_sensationalism])
spam_agent = LlmAgent(name="Spam_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_spam])
bert_agent = LlmAgent(name="BERT_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_BERT])
search_agent = LlmAgent(name="Web_Search_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_web_search])

manager_prompt = """
CRITICAL:
Your final message MUST be ONLY valid JSON.
Do not include markdown.
Do not include explanations outside JSON.
Do not include code fences.
You are the Factuality Root Manager. Your job is to generate a final fact-checking JSON report.
USE MULTI-TURN PERSISTENCE IF AGENT OUTPUTS ARE EMPTY OR INVALID!

### CRITICAL: MANDATORY DATA GATHERING
You MUST consult your sub-agents to gather data BEFORE generating your final response to the user. Do not stop or reply to the user until you have collected sufficient information from AT LEAST ONE of the following agents:

1. Call 'BERT_Agent' to get truthfulness probabilities.
2. Call 'Political_Bias_Agent' to check for stats and partisan framing.
3. Call 'Sensationalism_Agent' to get the emotional intensity score.
4. Call 'Spam_Agent' to check for bot-like characteristics.
5. Call 'Web_Search_Agent' to look for live evidence using keywords.

Wait for each agent to return its data, keep it in your internal memory, and immediately call the next agent on the list.

### FINAL SYNTHESIS
ONLY AFTER you have received data from all 6 agents, synthesize the results and output the final report.

[ORIGINAL FACTUALITY INSTRUCTIONS]
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION:
0-5: Probabilities for truthfulness classes (BERT_Agent).
7: Count of numeric/statistical entities.
8: Count of conservative bigram matches.
9: Count of liberal bigram matches.
10: Emotional intensity score (Sensationalism_Agent).
11: Spam likelihood score (Spam_Agent).

ANTI-BIAS CONSTRAINT:
- Treat predictive model scores only as auxiliary context. Rely on TEXTUAL EVIDENCE for final verdicts.

FACTUALITY FACTORS:
1. AUTHENTICITY (1–10): Verifiable details, sources, timestamps.
2. SENSATIONALISM (1–10): Density of hyperbole/drama.
3. POLITICAL BIAS (0–10 + tag): Partisan framing/selective omission.
4. SPAM (1–10): Bot-like content characteristics.
5. CONFIRMATION BIAS (1–10): Cherry-picked evidence.
6. SHORT-TERM UTILITY (1–10): Clickbait or monetization cues.

OUTPUT FORMAT (STRICT JSON):
{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "...",
  "factor_scores": [
    {"factor": "Authenticity", "score": 1-10, "reasoning": "..."},
    ...
  ]
}
"""

manager_agent = LlmAgent(name="FactCheck_Manager", model=manager_llm, instruction=manager_prompt,
    tools=[AgentTool(agent=bert_agent), AgentTool(agent=bias_agent), AgentTool(agent=sensational_agent), 
           AgentTool(agent=spam_agent), AgentTool(agent=search_agent)])

nest_asyncio.apply()
a2a_app = to_a2a(manager_agent)

# --- PORT LOGIC ---
PORT = 50008
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if is_port_in_use(PORT):
    logger.warning(f"Port {PORT} in use. Finding a new port...")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        PORT = s.getsockname()[1]

def run_server():
    uvicorn.run(a2a_app, host="0.0.0.0", port=PORT, log_level="error")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# --- COMMUNICATION & LOOP ---
def call_agent_api(text: str) -> str:
    send_payload = {"jsonrpc": "2.0", "method": "message/send", "id": 1,
        "params": {"message": {"messageId": str(uuid.uuid4()), "role": "user", "parts": [{"kind": "text", "text": text}]}}}
    try:
        r = requests.post(f"http://localhost:{PORT}/", json=send_payload, timeout=180)
        r.raise_for_status()
        res = r.json().get("result", {})
        for part in res.get("parts", []):
            if part.get("kind") == "text": return part["text"]
        return ""
    except Exception as e:
        logger.error(f"API Error: {e}")
        return ""

out_list = []
article_df = pd.read_csv("../data/labeled_articles.csv")
for i in range(article_df.shape[0]):
    news_snippet = article_df.iloc[i]['text']
    print(f"Analyzing article {i+1}...")
    
    final_verdict = call_agent_api(news_snippet)
    if not final_verdict:
        print("Empty response from API. Skipping.")
        break
        continue

    final_verdict_formatted = "============================================================\n" + final_verdict
    print("\nMANAGER FINAL DECISION:\n" + "="*60 + "\n" + final_verdict_formatted)
    
    try:
        raw_json = final_verdict_formatted.split('============================================================')[-1].strip()
        out_json = json.loads(raw_json)
        factor_dict = {fs["factor"].lower(): fs["score"] for fs in out_json["factor_scores"]}
        out_list.append(factor_dict)
    except Exception as e:
        print(f"Error parsing article {i}: {e}")
    break

eval_df = pd.DataFrame(out_list)
print("\nBATCH ANALYSIS COMPLETE")
print(eval_df)

C:\Users\Chris Mo\AppData\Local\Temp\ipykernel_30264\2514208238.py:135: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-flash-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-flash-preview') with Gemini(model='gemini-3-flash-preview'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  worker_llm = LiteLlm(model="gemini/gemini-3-flash-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
C:\Users\Chris Mo\AppData\Local\Temp\ipykernel_30264\2514208238.py:136: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-pro-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-pro-preview') with Gemini(model='gemini-3-pro-preview'). Set ADK_

Analyzing article 1...


c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\a2a\utils\agent_to_a2a.py:120: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service=InMemoryCredentialService(),
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\auth\credential_service\in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\a2a\executor\a2a_agent_executor.py:190: UserWarning: [EXPERIMENTAL] convert_a2a_request_to_agent_run_request: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and correspo

12:35:17 - LiteLLM:INFO: utils.py:3748 - 
LiteLLM completion() model= gemini-3-pro-preview; provider = gemini


2026-03-08 12:35:17,349 - INFO - 
LiteLLM completion() model= gemini-3-pro-preview; provider = gemini
2026-03-08 12:35:17,753 - ERROR - Error handling A2A request: litellm.BadRequestError: GeminiException BadRequestError - {
  "error": {
    "code": 400,
    "message": "API key expired. Please renew the API key.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "API_KEY_INVALID",
        "domain": "googleapis.com",
        "metadata": {
          "service": "generativelanguage.googleapis.com"
        }
      },
      {
        "@type": "type.googleapis.com/google.rpc.LocalizedMessage",
        "locale": "en-US",
        "message": "API key expired. Please renew the API key."
      }
    ]
  }
}
Traceback (most recent call last):
  File "c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2489, in asy


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Empty response from API. Skipping.

BATCH ANALYSIS COMPLETE
Empty DataFrame
Columns: []
Index: []
